# @tool과 Ch 4 함수 재사용

검증한 조회 함수에 도구 인터페이스를 더합니다.

## 준비

공개 CSV와 Ch 4 함수 원문을 사용합니다.

## 코드를 읽기 전에

`@tool`은 바로 아래 파이썬 함수를 LangChain의 도구 객체로 감쌉니다. 함수 이름과 문서 문자열, 타입 힌트에서 모델이 볼 이름·설명·입력 스키마를 만들지만 함수 본문을 모델에게 보여 주지는 않습니다. 일반 함수처럼 위치 인자를 넘기는 대신 도구 객체의 `invoke({"status": ...})`에 입력 사전을 전달하면 검증 뒤 함수가 실행되고 결과가 반환됩니다.

이 절은 Ch 4의 `fetch_orders()`를 다시 작성하지 않습니다. 모델이 선택할 `status`만 공개하고 CSV 경로는 애플리케이션 코드가 고정합니다. Before의 `TypeError`는 주문 자료의 문제가 아니라 도구 객체의 호출 형식을 지키지 않은 인터페이스 오류입니다.

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


### 다음 셀을 읽는 방법

다음 셀은 Ch 4에서 검증한 `fetch_orders(csv_path, status=...)`의 본문을 같은 입력·반환 형식으로 다시 선언합니다. CSV 경로와 선택 상태를 받아 각 행을 사전으로 담은 목록을 반환하며, 아직 LangChain 도구를 만들거나 모델을 호출하지 않습니다.


In [2]:
# Ch 4의 검증한 조회 함수 본문을 그대로 사용합니다.
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")


## 도구 정의

In [3]:
# 모델에 공개할 status 인자만 도구로 감쌉니다.
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)


## 같은 값 비교

In [4]:
# 직접 호출과 도구 호출의 실제 주문 행을 비교합니다.
direct = fetch_orders(data_path, status="cancelled")
wrapped = list_orders_by_status.invoke({"status": "cancelled"})
print({"same": direct == wrapped, "rows": wrapped})
assert direct == wrapped and wrapped[0]["order_id"] == "O104"


{'same': True, 'rows': [{'order_id': 'O104', 'customer_id': 'C003', 'status': 'cancelled', 'amount': 9000}]}


## 실패 재현

In [5]:
# 도구 객체에 일반 함수 호출 문법을 쓰면 발생하는 오류를 확인합니다.
try:
    list_orders_by_status("paid")
except TypeError as error:
    print(type(error).__name__, str(error))


TypeError 'StructuredTool' object is not callable


## 수정

In [6]:
# invoke에 인자 사전을 전달해 paid 주문 두 건을 조회합니다.
result = list_orders_by_status.invoke({"status": "paid"})
print({"count": len(result), "first_id": result[0]["order_id"]})
assert len(result) == 2 and {r["order_id"] for r in result} == {"O101", "O103"}


{'count': 2, 'first_id': 'O101'}


## 다른 상황에 적용하기

재고 조회 함수를 도구로 만들 때 창고 ID는 모델이 골라야 할 업무 인자일 수 있지만, 데이터베이스 연결 문자열은 프로그램 설정으로 남겨야 합니다. 모델에게 공개할 입력을 줄이면 선택 오류와 권한 범위가 함께 줄어듭니다. 기존 함수를 재사용한다는 말은 기존 검증과 테스트를 그대로 활용한다는 뜻이며, 도구 장식만으로 원래 함수의 데이터 품질이 좋아지지는 않습니다.